# LoRa Predictive and Optimization Model

### Configuration and Data Variables

In [ ]:
# Load environment variables
load_dotenv()
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('lora_system.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

@dataclass
class LoRaParameters:
    """LoRa communication parameters"""
    tx_power: float = 14.0  # dBm
    spreading_factor: int = 7
    frequency: float = 868.0  # MHz
    bandwidth: float = 125.0  # kHz
    coding_rate: int = 4  # 4/5

@dataclass
class PathPoint:
    """Represents a point in the path with all attributes"""
    lat: float
    lon: float
    elevation: float = 0.0
    land_cover: int = 50
    terrain_penalty: float = 0.3
    rssi: float = -100.0
    snr: float = 0.0
    pdr: float = 0.5
    path_loss: float = 100.0
    distance_to_start: float = 0.0
    distance_to_destination: float = 0.0
    grid_x: int = 0
    grid_y: int = 0
    is_relay: bool = False
    hop_number: int = 0

@dataclass
class OptimizationConfig:
    """Configuration for path optimization"""
    grid_spacing_km: float = 1.5
    max_hop_distance_km: float = 5.0
    min_relay_distance_km: float = 1.0
    interpolate_between_points: bool = True
    use_real_data: bool = True
    adaptive_grid: bool = True
    path_smoothing: bool = True


### 1. Google Earth Engine Integration

In [ ]:
class GoogleEarthEngineIntegration:
    """Robust real-time spatial data fetching from Google Earth Engine"""
    
    def __init__(self, use_service_account=False, service_account_key=None):
        self.initialized = False
        self.cache = {}
        self.rate_limiter = RateLimiter(calls_per_second=10)
        self.use_fallback = True  # Force fallback mode by default
        
        try:
            # Try to initialize with explicit error handling
            try:
                # First, try to initialize with a project ID if available
                project_id = os.getenv('GEE_PROJECT_ID')
                if project_id:
                    ee.Initialize(project=project_id)
                    logger.info(f"Google Earth Engine initialized with project: {project_id}")
                else:
                    # Try without project ID
                    ee.Initialize()
                    logger.info("Google Earth Engine initialized without project ID")
                
                # Test initialization with a simple request
                test_point = ee.Geometry.Point([0, 0])
                test_result = ee.Image('USGS/SRTMGL1_003').sample(test_point, scale=30).getInfo()
                
                if test_result and len(test_result['features']) > 0:
                    logger.info("Google Earth Engine test successful!")
                    self.initialized = True
                    self.use_fallback = False  # Only use GEE if it's working
                else:
                    logger.warning("Google Earth Engine test returned empty results")
                    self.initialized = False
                    
            except Exception as init_error:
                logger.warning(f"Google Earth Engine initialization failed: {init_error}")
                self.initialized = False
        
        except Exception as e:
            logger.warning(f"Could not initialize Google Earth Engine: {e}")
            self.initialized = False
        
        # Land cover penalty mapping
        self.land_cover_penalties = {
            10: 0.4, 20: 0.3, 30: 0.2, 40: 0.25, 50: 0.4,
            60: 0.15, 70: 0.05, 80: 0.0, 90: 0.35, 95: 0.3, 100: 0.1
        }
    
    def get_elevation(self, lat: float, lon: float, retry_count=3) -> float:
        """Fetch real elevation data from SRTM (30m resolution) with retry logic"""
        if not self.initialized or self.use_fallback:
            return self._simulate_elevation(lat, lon)
        
        cache_key = f"elevation_{lat:.6f}_{lon:.6f}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        for attempt in range(retry_count):
            try:
                with self.rate_limiter:
                    point = ee.Geometry.Point([lon, lat])
                    srtm = ee.Image('USGS/SRTMGL1_003')
                    
                    elevation_dict = srtm.reduceRegion(
                        reducer=ee.Reducer.first(),
                        geometry=point,
                        scale=30,
                        maxPixels=1
                    ).getInfo()
                    
                    elevation = elevation_dict.get('elevation')
                    
                    if elevation is not None:
                        elevation = float(elevation)
                        self.cache[cache_key] = elevation
                        return elevation
                    else:
                        return self._simulate_elevation(lat, lon)
                        
            except Exception as e:
                if attempt == retry_count - 1:
                    logger.warning(f"Failed to get elevation for ({lat}, {lon}): {e}")
                    return self._simulate_elevation(lat, lon)
                time.sleep(0.5)
        
        return self._simulate_elevation(lat, lon)
    
    def get_land_cover(self, lat: float, lon: float, retry_count=3) -> Tuple[int, float]:
        """Fetch real land cover data from ESA WorldCover (10m resolution)"""
        if not self.initialized or self.use_fallback:
            return self._simulate_land_cover(lat, lon)
        
        cache_key = f"landcover_{lat:.6f}_{lon:.6f}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        for attempt in range(retry_count):
            try:
                with self.rate_limiter:
                    point = ee.Geometry.Point([lon, lat])
                    worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
                    
                    lc_dict = worldcover.reduceRegion(
                        reducer=ee.Reducer.first(),
                        geometry=point,
                        scale=10,
                        maxPixels=1
                    ).getInfo()
                    
                    land_cover = lc_dict.get('Map')
                    
                    if land_cover is not None:
                        land_cover_code = int(land_cover)
                        terrain_penalty = self.land_cover_penalties.get(land_cover_code, 0.5)
                        result = (land_cover_code, terrain_penalty)
                        self.cache[cache_key] = result
                        return result
                    else:
                        return 80, 0.0  # Water body with no penalty
                        
            except Exception as e:
                if attempt == retry_count - 1:
                    logger.warning(f"Failed to get land cover for ({lat}, {lon}): {e}")
                    return self._simulate_land_cover(lat, lon)
                time.sleep(0.5)
        
        return self._simulate_land_cover(lat, lon)
    
    def get_spatial_features(self, lat: float, lon: float) -> Dict:
        """Fetch all spatial features for a location with caching"""
        cache_key = f"spatial_{lat:.6f}_{lon:.6f}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        elevation = self.get_elevation(lat, lon)
        land_cover, terrain_penalty = self.get_land_cover(lat, lon)
        
        result = {
            'elevation': elevation,
            'land_cover': land_cover,
            'terrain_penalty': terrain_penalty
        }
        self.cache[cache_key] = result
        return result
    
    def batch_get_spatial_features(self, coordinates: List[Tuple[float, float]], 
                                   batch_size: int = 50,
                                   use_batch_mode: bool = True) -> List[Dict]:
        """Fetch spatial features for multiple locations with optimized batching"""
        total = len(coordinates)
        logger.info(f"Fetching spatial features for {total} locations...")
        
        if use_batch_mode and self.initialized and not self.use_fallback:
            return self._batch_get_spatial_features_efficient(coordinates)
        else:
            logger.info("Using sequential mode (GEE not available or disabled)")
            return self._batch_get_spatial_features_sequential(coordinates, batch_size)
    
    def _batch_get_spatial_features_efficient(self, coordinates: List[Tuple[float, float]]) -> List[Dict]:
        """Efficient batch fetching using GEE's batch capabilities"""
        results = []
        errors = 0
        successes = 0
        
        try:
            # Create points as a FeatureCollection
            features = []
            for i, (lat, lon) in enumerate(coordinates):
                point = ee.Geometry.Point([lon, lat])
                features.append(ee.Feature(point, {'index': i}))
            
            points_fc = ee.FeatureCollection(features)
            
            # Sample elevation
            srtm = ee.Image('USGS/SRTMGL1_003')
            elevation_sampled = srtm.reduceRegions(
                collection=points_fc,
                reducer=ee.Reducer.first(),
                scale=30
            )
            
            # Sample land cover
            worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
            landcover_sampled = worldcover.reduceRegions(
                collection=elevation_sampled,
                reducer=ee.Reducer.first(),
                scale=10
            )
            
            # Get results
            sampled_data = landcover_sampled.getInfo()
            
            # Process results
            result_dict = {}
            for feature in sampled_data['features']:
                idx = feature['properties']['index']
                elevation = feature['properties'].get('elevation', None)
                land_cover = feature['properties'].get('Map', None)
                
                if elevation is None:
                    elevation = self._simulate_elevation(coordinates[idx][0], coordinates[idx][1])
                else:
                    elevation = float(elevation)
                    successes += 1
                
                if land_cover is None:
                    land_cover = 50
                    terrain_penalty = 0.3
                    errors += 1
                else:
                    land_cover = int(land_cover)
                    terrain_penalty = self.land_cover_penalties.get(land_cover, 0.5)
                    successes += 1
                
                result_dict[idx] = {
                    'elevation': elevation,
                    'land_cover': land_cover,
                    'terrain_penalty': terrain_penalty,
                    'latitude': coordinates[idx][0],
                    'longitude': coordinates[idx][1]
                }
            
            # Ensure all coordinates have results in order
            for i in range(len(coordinates)):
                if i in result_dict:
                    results.append(result_dict[i])
                else:
                    # Missing data, use simulation
                    lat, lon = coordinates[i]
                    results.append({
                        'elevation': self._simulate_elevation(lat, lon),
                        'land_cover': 50,
                        'terrain_penalty': 0.3,
                        'latitude': lat,
                        'longitude': lon
                    })
                    errors += 1
            
            logger.info(f"Batch fetch completed - Successes: {successes}, Errors/Missing: {errors}")
            
        except Exception as e:
            logger.warning(f"Batch mode failed ({e}), falling back to sequential mode...")
            return self._batch_get_spatial_features_sequential(coordinates, 50)
        
        return results
    
    def _batch_get_spatial_features_sequential(self, coordinates: List[Tuple[float, float]], 
                                               batch_size: int = 50) -> List[Dict]:
        """Sequential fetching with progress tracking and rate limiting"""
        results = []
        total = len(coordinates)
        errors = 0
        successes = 0
        
        for i, (lat, lon) in enumerate(coordinates):
            if i > 0 and i % batch_size == 0:
                logger.info(f"Progress: {i}/{total} ({i/total*100:.1f}%) - Successes: {successes}, Errors: {errors}")
                time.sleep(1)  # Rate limiting
            
            try:
                features = self.get_spatial_features(lat, lon)
                features['latitude'] = lat
                features['longitude'] = lon
                results.append(features)
                successes += 1
            except Exception as e:
                # If total failure, use simulation
                features = {
                    'elevation': self._simulate_elevation(lat, lon),
                    'land_cover': 50,
                    'terrain_penalty': 0.3,
                    'latitude': lat,
                    'longitude': lon
                }
                results.append(features)
                errors += 1
        
        logger.info(f"Completed: {total}/{total} (100%) - Successes: {successes}, Errors: {errors}")
        
        if errors > 0:
            logger.warning(f"{errors} locations used simulated data (GEE data unavailable)")
        
        return results
    
    def _simulate_elevation(self, lat: float, lon: float) -> float:
        """Simulate elevation when GEE is not available"""
        return abs(np.sin(lat * 10) * np.cos(lon * 10) * 500)
    
    def _simulate_land_cover(self, lat: float, lon: float) -> Tuple[int, float]:
        """Simulate land cover when GEE is not available"""
        land_cover_codes = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
        land_cover = np.random.choice(land_cover_codes)
        terrain_penalty = self.land_cover_penalties.get(land_cover, 0.5)
        return land_cover, terrain_penalty

class RateLimiter:
    """Simple rate limiter for API calls"""
    def __init__(self, calls_per_second=10):
        self.calls_per_second = calls_per_second
        self.last_call = 0
        
    def __enter__(self):
        elapsed = time.time() - self.last_call
        if elapsed < 1.0 / self.calls_per_second:
            time.sleep((1.0 / self.calls_per_second) - elapsed)
        self.last_call = time.time()
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        pass


### 2. Data Loading and Preprocessing

In [ ]:
class LoRaDataPreprocessor:
    """Robust data loading and preprocessing with flexible format handling"""
    
    def __init__(self, gee_integration: Optional[GoogleEarthEngineIntegration] = None):
        self.scaler = StandardScaler()
        self.gee = gee_integration
        
    def load_dataset1(self, filepath):
        """Load dataset with format: latitude,longitude,elevation,land_cover,etc."""
        try:
            # Try different separators
            for sep in [',', '|', '\t']:
                try:
                    df = pd.read_csv(filepath, sep=sep)
                    if len(df.columns) > 5:  # Reasonable number of columns
                        break
                except:
                    continue
            else:
                raise ValueError("Could not determine file format")
            
            # Map column names flexibly
            column_mapping = {
                'latitude': ['latitude', 'lat'],
                'longitude': ['longitude', 'lon'],
                'elevation': ['elevation', 'altitude', 'elev'],
                'land_cover': ['land_cover', 'land_cover_code', 'landcover'],
                'terrain_penalty': ['terrain_penalty', 'terrain'],
                'RSSI': ['RSSI', 'rssi'],
                'SNR': ['SNR', 'snr'],
                'PDR': ['PDR', 'pdr'],
                'observed_path_loss': ['observed_path_loss', 'path_loss', 'loss'],
                'spreading_factor': ['spreading_factor', 'sf'],
                'frequency': ['frequency', 'freq'],
                'tx_power': ['tx_power', 'power'],
                'distance_to_start': ['distance_to_start'],
                'distance_to_destination': ['distance_to_destination']
            }
            
            # Create standardized dataframe
            df_processed = pd.DataFrame()
            for std_col, possible_cols in column_mapping.items():
                for col in possible_cols:
                    if col in df.columns:
                        df_processed[std_col] = df[col]
                        break
                else:
                    # Set default values if column not found
                    if std_col == 'spreading_factor':
                        df_processed[std_col] = 7
                    elif std_col == 'frequency':
                        df_processed[std_col] = 868
                    elif std_col == 'tx_power':
                        df_processed[std_col] = 14
                    elif std_col in ['terrain_penalty']:
                        df_processed[std_col] = 0.3
                    else:
                        df_processed[std_col] = 0
            
            return df_processed.dropna(subset=['RSSI', 'SNR', 'PDR'])
        
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            return pd.DataFrame()
    
    def load_dataset2(self, filepath):
        """Load dataset with format: device_id,gateway_id,latitude,longitude,altitude,etc."""
        try:
            # Try different separators
            for sep in [',', '|', '\t']:
                try:
                    df = pd.read_csv(filepath, sep=sep)
                    if len(df.columns) > 5:  # Reasonable number of columns
                        break
                except:
                    continue
            else:
                raise ValueError("Could not determine file format")
            
            # Map column names flexibly
            column_mapping = {
                'latitude': ['latitude', 'lat'],
                'longitude': ['longitude', 'lon'],
                'elevation': ['altitude', 'elevation', 'elev'],
                'land_cover': ['land_cover_code', 'land_cover', 'landcover'],
                'terrain_penalty': ['terrain_penalty', 'terrain'],
                'RSSI': ['RSSI', 'rssi'],
                'SNR': ['SNR', 'snr'],
                'PDR': ['PDR', 'pdr'],
                'observed_path_loss': ['observed_path_loss', 'path_loss', 'loss'],
                'spreading_factor': ['spreading_factor', 'sf'],
                'frequency': ['frequency', 'freq'],
                'tx_power': ['tx_power', 'power'],
                'distance_to_start': ['distance_to_start'],
                'distance_to_destination': ['distance_to_destination']
            }
            
            # Create standardized dataframe
            df_processed = pd.DataFrame()
            for std_col, possible_cols in column_mapping.items():
                for col in possible_cols:
                    if col in df.columns:
                        df_processed[std_col] = df[col]
                        break
                else:
                    # Set default values if column not found
                    if std_col == 'spreading_factor':
                        df_processed[std_col] = 7
                    elif std_col == 'frequency':
                        df_processed[std_col] = 868
                    elif std_col == 'tx_power':
                        df_processed[std_col] = 14
                    elif std_col in ['terrain_penalty']:
                        df_processed[std_col] = 0.3
                    else:
                        df_processed[std_col] = 0
            
            return df_processed.dropna(subset=['RSSI', 'SNR', 'PDR'])
        
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            return pd.DataFrame()
    
    def merge_datasets(self, df1, df2):
        """Merge and clean datasets with validation"""
        if df1.empty and df2.empty:
            raise ValueError("Both datasets are empty!")
        
        if df1.empty:
            df_combined = df2.copy()
        elif df2.empty:
            df_combined = df1.copy()
        else:
            df_combined = pd.concat([df1, df2], ignore_index=True)
        
        # Drop rows with critical missing values
        df_combined = df_combined.dropna(subset=['RSSI', 'SNR', 'PDR'])
        
        # Feature engineering
        df_combined['distance_total'] = df_combined['distance_to_start'] + df_combined['distance_to_destination']
        df_combined['elevation_normalized'] = df_combined['elevation'] / 1000
        df_combined['rssi_normalized'] = (df_combined['RSSI'] + 130) / 100
        
        # Add default values if columns don't exist
        if 'frequency' not in df_combined.columns:
            df_combined['frequency'] = 868  # Default EU868
            logger.warning("⚠️  'frequency' column not found, using default 868 MHz")
        
        if 'tx_power' not in df_combined.columns:
            df_combined['tx_power'] = 14  # Default 14 dBm
            logger.warning("⚠️  'tx_power' column not found, using default 14 dBm")
        
        logger.info(f"Combined dataset shape: {df_combined.shape}")
        logger.info(f"Missing values:\n{df_combined.isnull().sum()}")
        
        return df_combined
    
    def prepare_features(self, df, target_cols=['RSSI', 'SNR', 'PDR', 'observed_path_loss']):
        """Prepare feature and target matrices with validation"""
        feature_cols = [
            'elevation', 'land_cover', 'terrain_penalty',
            'distance_to_start', 'distance_to_destination',
            'spreading_factor', 'frequency', 'tx_power',
            'distance_total', 'elevation_normalized'
        ]
        
        # Validate all feature columns exist
        missing_cols = [col for col in feature_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing feature columns: {missing_cols}")
        
        X = df[feature_cols].values
        y = df[target_cols].values
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        logger.info(f"Training samples: {len(X_train)}")
        logger.info(f"Test samples: {len(X_test)}")
        logger.info(f"Features: {len(feature_cols)}")
        
        return X_train_scaled, X_test_scaled, y_train, y_test, feature_cols


### 3. Pytorch Neural Network Model

In [ ]:
class LoRaDataset(Dataset):
    """PyTorch dataset for LoRa data"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LoRaNeuralNetwork(nn.Module):
    """Advanced neural network with configurable architecture"""
    def __init__(self, input_size, output_size, config=None):
        super(LoRaNeuralNetwork, self).__init__()
        
        # Default configuration
        default_config = {
            'hidden_sizes': [256, 128, 64, 32],
            'dropout_rate': 0.3,
            'activation': 'relu',
            'batch_norm': True,
            'residual_connections': True
        }
        
        if config:
            default_config.update(config)
        
        self.config = default_config
        layers = []
        prev_size = input_size
        
        for i, hidden_size in enumerate(self.config['hidden_sizes']):
            layers.append(nn.Linear(prev_size, hidden_size))
            
            if self.config['batch_norm']:
                layers.append(nn.BatchNorm1d(hidden_size))
            
            if self.config['activation'] == 'relu':
                layers.append(nn.ReLU())
            elif self.config['activation'] == 'leaky_relu':
                layers.append(nn.LeakyReLU(0.2))
            elif self.config['activation'] == 'elu':
                layers.append(nn.ELU())
            
            if self.config['dropout_rate'] > 0:
                layers.append(nn.Dropout(self.config['dropout_rate']))
            
            # Add residual connection for deeper networks
            if self.config['residual_connections'] and i > 0 and prev_size == hidden_size:
                layers.append(ResidualConnection())
            
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

    def predict(self, X):
        """Make predictions with the model"""
        self.eval()
        with torch.no_grad():
            if isinstance(X, np.ndarray):
                X = torch.FloatTensor(X)
            X = X.to(next(self.parameters()).device)
            predictions = self.forward(X).cpu().numpy()
        return predictions

class ResidualConnection(nn.Module):
    """Residual connection for deeper networks"""
    def __init__(self):
        super(ResidualConnection, self).__init__()
        
    def forward(self, x):
        return x

class NeuralNetworkTrainer:
    """Advanced neural network trainer with early stopping and learning rate scheduling"""
    def __init__(self, input_size, output_size, device, config=None):
        self.device = device
        self.config = config or {}
        
        # Model configuration
        model_config = self.config.get('model', {})
        self.model = LoRaNeuralNetwork(input_size, output_size, model_config).to(device)
        
        # Training configuration
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(
            self.model.parameters(), 
            lr=self.config.get('learning_rate', 0.001),
            weight_decay=self.config.get('weight_decay', 1e-5)
        )
        
        # Learning rate scheduling
        scheduler_type = self.config.get('scheduler', 'reduce_on_plateau')
        if scheduler_type == 'reduce_on_plateau':
            self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer, mode='min', patience=10, factor=0.5, verbose=True
            )
        elif scheduler_type == 'cosine':
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer, T_max=self.config.get('epochs', 100)
            )
        
        # Early stopping
        self.early_stopping_patience = self.config.get('early_stopping_patience', 20)
        self.early_stopping_counter = 0
        self.best_val_loss = float('inf')
        
        self.train_losses = []
        self.val_losses = []
        
    def train(self, train_loader, val_loader, epochs=None):
        """Train the neural network with early stopping"""
        epochs = epochs or self.config.get('epochs', 100)
        logger.info(f"\nTraining Neural Network on {self.device}...")
        logger.info(f"Model architecture: {self.model.config}")
        logger.info(f"Training configuration: {self.config}")
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0
            train_steps = 0
            
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                
                self.optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = self.criterion(outputs, y_batch)
                loss.backward()
                
                # Gradient clipping
                if self.config.get('gradient_clip', 0) > 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.config['gradient_clip'])
                
                self.optimizer.step()
                
                train_loss += loss.item()
                train_steps += 1
            
            train_loss /= train_steps
            self.train_losses.append(train_loss)
            
            # Validation phase
            self.model.eval()
            val_loss = 0
            val_steps = 0
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                    outputs = self.model(X_batch)
                    loss = self.criterion(outputs, y_batch)
                    val_loss += loss.item()
                    val_steps += 1
            
            val_loss /= val_steps
            self.val_losses.append(val_loss)
            
            # Learning rate scheduling
            if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                self.scheduler.step(val_loss)
            else:
                self.scheduler.step()
            
            # Early stopping
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.early_stopping_counter = 0
                # Save best model
                torch.save(self.model.state_dict(), 'best_model.pth')
            else:
                self.early_stopping_counter += 1
                if self.early_stopping_counter >= self.early_stopping_patience:
                    logger.info(f"Early stopping at epoch {epoch+1}")
                    # Load best model
                    self.model.load_state_dict(torch.load('best_model.pth'))
                    break
            
            if (epoch + 1) % 10 == 0:
                logger.info(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        logger.info("Neural Network training completed!")
        
    def predict(self, X):
        """Make predictions with the trained model"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.model(X_tensor).cpu().numpy()
        return predictions


### 4. Random Forest Model

In [ ]:
class RandomForestModel:
    """Random Forest model with feature importance"""
    def __init__(self, n_estimators=100):
        self.models = {
            'RSSI': RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1),
            'SNR': RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1),
            'PDR': RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1),
            'path_loss': RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1)
        }
        
    def train(self, X_train, y_train):
        """Train all Random Forest models"""
        logger.info("\nTraining Random Forest models...")
        for i, (name, model) in enumerate(self.models.items()):
            logger.info(f"Training {name} model...")
            model.fit(X_train, y_train[:, i])
        logger.info("Random Forest training completed!")
        
    def predict(self, X):
        """Make predictions with all models"""
        predictions = np.zeros((X.shape[0], len(self.models)))
        for i, model in enumerate(self.models.values()):
            predictions[:, i] = model.predict(X)
        return predictions
    
    def get_feature_importance(self, feature_names):
        """Get feature importance for all models"""
        importance_dict = {}
        for name, model in self.models.items():
            importance_dict[name] = dict(zip(feature_names, model.feature_importances_))
        return importance_dict


### 5. XGBoost Model

In [ ]:
class XGBoostModel:
    """XGBoost model with advanced configuration"""
    def __init__(self):
        self.models = {
            'RSSI': xgb.XGBRegressor(
                n_estimators=100, 
                learning_rate=0.1, 
                max_depth=6, 
                random_state=42,
                n_jobs=-1
            ),
            'SNR': xgb.XGBRegressor(
                n_estimators=100, 
                learning_rate=0.1, 
                max_depth=6, 
                random_state=42,
                n_jobs=-1
            ),
            'PDR': xgb.XGBRegressor(
                n_estimators=100, 
                learning_rate=0.1, 
                max_depth=6, 
                random_state=42,
                n_jobs=-1
            ),
            'path_loss': xgb.XGBRegressor(
                n_estimators=100, 
                learning_rate=0.1, 
                max_depth=6, 
                random_state=42,
                n_jobs=-1
            )
        }
        
    def train(self, X_train, y_train):
        """Train all XGBoost models"""
        logger.info("\nTraining XGBoost models...")
        for i, (name, model) in enumerate(self.models.items()):
            logger.info(f"Training {name} model...")
            model.fit(X_train, y_train[:, i])
        logger.info("XGBoost training completed!")
        
    def predict(self, X):
        """Make predictions with all models"""
        predictions = np.zeros((X.shape[0], len(self.models)))
        for i, model in enumerate(self.models.values()):
            predictions[:, i] = model.predict(X)
        return predictions


### 6. Path Optimization with A* Algorithm

In [ ]:
class PathOptimizer:
    """Advanced path optimization with adaptive grid and terrain interpolation"""
    
    def __init__(self, model, scaler, feature_cols, gee_integration: GoogleEarthEngineIntegration):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.gee = gee_integration
        
        # LoRa-specific parameters
        self.lora_params = {
            'sensitivity': {7: -123, 8: -126, 9: -129, 10: -132, 11: -134, 12: -137},
            'max_distance': {7: 15000, 8: 7500, 9: 3750, 10: 1875, 11: 937, 12: 468}
        }
        
    def calculate_distance(self, lat1, lon1, lat2, lon2):
        """Calculate Haversine distance in meters"""
        R = 6371000
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        
        a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        
        return R * c
    
    def generate_adaptive_grid(self, start_lat, start_lon, dest_lat, dest_lon, 
                              config: OptimizationConfig):
        """Generate adaptive grid based on terrain complexity"""
        # Calculate total distance
        total_distance = self.calculate_distance(start_lat, start_lon, dest_lat, dest_lon)
        
        if config.adaptive_grid:
            # Sample terrain along direct path to determine complexity
            num_samples = 10
            lats = np.linspace(start_lat, dest_lat, num_samples)
            lons = np.linspace(start_lon, dest_lon, num_samples)
            
            elevation_variance = 0
            terrain_penalty_sum = 0
            
            for lat, lon in zip(lats, lons):
                spatial = self.gee.get_spatial_features(lat, lon)
                elevation_variance += spatial['elevation'] ** 2
                terrain_penalty_sum += spatial['terrain_penalty']
            
            # Adjust grid spacing based on complexity
            complexity_score = (elevation_variance / num_samples) / 1000 + terrain_penalty_sum / num_samples
            adjusted_spacing = config.grid_spacing_km / (1 + complexity_score)
        else:
            adjusted_spacing = config.grid_spacing_km
        
        # Generate grid with adjusted spacing
        num_points_axis = max(5, int(np.ceil(total_distance / (adjusted_spacing * 1000))))
        
        lats = np.linspace(start_lat, dest_lat, num_points_axis)
        lons = np.linspace(start_lon, dest_lon, num_points_axis)
        
        grid_points = []
        coordinates = []
        
        for i, lat in enumerate(lats):
            for j, lon in enumerate(lons):
                grid_points.append(PathPoint(
                    lat=lat,
                    lon=lon,
                    grid_x=i,
                    grid_y=j
                ))
                coordinates.append((lat, lon))
        
        actual_spacing = self.calculate_distance(lats[0], lons[0], lats[1], lons[1]) / 1000
        logger.info(f"  Total distance: {total_distance/1000:.2f} km")
        logger.info(f"  Grid spacing: {adjusted_spacing:.2f} km (actual: {actual_spacing:.2f} km)")
        logger.info(f"  Grid size: {num_points_axis}x{num_points_axis} = {num_points_axis**2} points")
        
        return grid_points, coordinates, num_points_axis
    
    def interpolate_spatial_features(self, point1: PathPoint, point2: PathPoint, num_samples=5):
        """Sample spatial features between two points to detect micro-topography"""
        # Generate intermediate points
        lats = np.linspace(point1.lat, point2.lat, num_samples)
        lons = np.linspace(point1.lon, point2.lon, num_samples)
        
        elevations = []
        land_covers = []
        terrain_penalties = []
        
        for lat, lon in zip(lats, lons):
            spatial = self.gee.get_spatial_features(lat, lon)
            elevations.append(spatial['elevation'])
            land_covers.append(spatial['land_cover'])
            terrain_penalties.append(spatial['terrain_penalty'])
        
        # Calculate statistics
        return {
            'avg_elevation': np.mean(elevations),
            'max_elevation': np.max(elevations),
            'min_elevation': np.min(elevations),
            'elevation_variance': np.var(elevations),
            'elevation_range': np.max(elevations) - np.min(elevations),
            'avg_terrain_penalty': np.mean(terrain_penalties),
            'max_terrain_penalty': np.max(terrain_penalties),
            'dominant_land_cover': int(np.median(land_covers))
        }
    
    def predict_at_point(self, point: PathPoint, start_lat, start_lon, dest_lat, dest_lon,
                         lora_params: LoRaParameters, use_interpolation=False, 
                         neighbor_point: Optional[PathPoint] = None):
        """Predict communication metrics at a point using REAL spatial features"""
        point.distance_to_start = self.calculate_distance(
            point.lat, point.lon, start_lat, start_lon
        )
        point.distance_to_destination = self.calculate_distance(
            point.lat, point.lon, dest_lat, dest_lon
        )
        distance_total = point.distance_to_start + point.distance_to_destination
        
        # Use interpolated features if available
        if use_interpolation and neighbor_point is not None:
            interp_features = self.interpolate_spatial_features(point, neighbor_point, num_samples=5)
            
            # Use max terrain penalty and elevation range for more conservative estimates
            point.elevation = interp_features['avg_elevation']
            point.terrain_penalty = interp_features['max_terrain_penalty']
            point.land_cover = interp_features['dominant_land_cover']
            
            # Store interpolation metadata
            point.elevation_range = interp_features['elevation_range']
            point.elevation_variance = interp_features['elevation_variance']
        else:
            spatial = self.gee.get_spatial_features(point.lat, point.lon)
            point.elevation = spatial['elevation']
            point.land_cover = spatial['land_cover']
            point.terrain_penalty = spatial['terrain_penalty']
            point.elevation_range = 0
            point.elevation_variance = 0
        
        elevation_normalized = point.elevation / 1000
        
        # Create feature vector with REAL spatial data (10 features)
        features = np.array([[
            point.elevation,
            point.land_cover,
            point.terrain_penalty,
            point.distance_to_start,
            point.distance_to_destination,
            lora_params.spreading_factor,
            lora_params.frequency,
            lora_params.tx_power,
            distance_total,
            elevation_normalized
        ]])
        
        # Scale features
        features_scaled = self.scaler.transform(features)
        
        # Predict
        predictions = self.model.predict(features_scaled)[0]
        
        # Clamp predictions to physically plausible values
        point.rssi = np.clip(predictions[0], -150, -20)  # RSSI between -150 and -20 dBm
        point.snr = predictions[1]  # SNR can be negative
        point.pdr = np.clip(predictions[2], 0, 1)      # PDR between 0 and 1
        point.path_loss = predictions[3]
        
        return point
    
    def calculate_lora_cost(self, point: PathPoint, distance: float, lora_params: LoRaParameters):
        """Calculate path cost based on LoRa-specific characteristics"""
        # Get LoRa parameters
        sensitivity = self.lora_params['sensitivity'].get(lora_params.spreading_factor, -120)
        max_distance = self.lora_params['max_distance'].get(lora_params.spreading_factor, 5000)
        
        # Calculate link budget
        tx_power = lora_params.tx_power
        link_budget = tx_power - point.path_loss
        
        # Calculate reliability metrics
        margin = link_budget - sensitivity
        reliability = 1 / (1 + np.exp(-margin / 5))  # Sigmoid function for reliability
        
        # Distance penalty (exponential for LoRa)
        distance_penalty = np.exp(distance / max_distance) - 1
        
        # PDR cost
        pdr_cost = (1 - max(0, min(1, point.pdr))) * 0.5
        
        # Combined cost
        cost = pdr_cost + distance_penalty * 0.3 + (1 - reliability) * 0.2
        
        return cost
    
    def sample_direct_path(self, start_lat, start_lon, dest_lat, dest_lon, 
                           lora_params: LoRaParameters, num_samples=10):
        """Sample points along the direct path for evaluation"""
        lats = np.linspace(start_lat, dest_lat, num_samples)
        lons = np.linspace(start_lon, dest_lon, num_samples)
        
        direct_points = []
        for lat, lon in zip(lats, lons):
            point = PathPoint(lat=lat, lon=lon)
            spatial_features = self.gee.get_spatial_features(lat, lon)
            point = self.predict_at_point(
                point, start_lat, start_lon, dest_lat, dest_lon,
                lora_params
            )
            direct_points.append(point)
        
        # Calculate average metrics
        avg_rssi = np.mean([p.rssi for p in direct_points])
        avg_snr = np.mean([p.snr for p in direct_points])
        avg_pdr = np.mean([p.pdr for p in direct_points])
        avg_path_loss = np.mean([p.path_loss for p in direct_points])
        
        return {
            'RSSI': avg_rssi,
            'SNR': avg_snr,
            'PDR': avg_pdr,
            'path_loss': avg_path_loss,
            'points': direct_points
        }
    
    def smooth_path(self, path: List[PathPoint]):
        """Apply path smoothing to generate more practical routes"""
        if len(path) <= 2:
            return path
        
        # Simple moving average smoothing
        smoothed_path = []
        window_size = min(3, len(path) // 2)
        
        for i in range(len(path)):
            start_idx = max(0, i - window_size // 2)
            end_idx = min(len(path), i + window_size // 2 + 1)
            
            avg_lat = np.mean([path[j].lat for j in range(start_idx, end_idx)])
            avg_lon = np.mean([path[j].lon for j in range(start_idx, end_idx)])
            
            # Create smoothed point
            smoothed_point = PathPoint(
                lat=avg_lat,
                lon=avg_lon,
                **{k: v for k, v in path[i].__dict__.items() if k not in ['lat', 'lon']}
            )
            smoothed_path.append(smoothed_point)
        
        return smoothed_path
    
    def find_optimal_path(self, start_lat, start_lon, dest_lat, dest_lon,
                          lora_params: LoRaParameters,
                          config: OptimizationConfig):
        """A* pathfinding algorithm with REAL-TIME Google Earth Engine data"""
        logger.info("\n" + "="*60)
        logger.info("PATH OPTIMIZATION WITH REAL-TIME SPATIAL DATA")
        logger.info("="*60)
        
        # Generate adaptive grid
        grid_points, coordinates, num_axis_points = self.generate_adaptive_grid(
            start_lat, start_lon, dest_lat, dest_lon, config
        )
        
        logger.info(f"  Estimated API calls: {len(grid_points)} (main grid)")
        
        # Fetch REAL spatial features from Google Earth Engine
        if config.use_real_data:
            logger.info("\nFetching REAL spatial data from Google Earth Engine...")
            logger.info("  - Elevation from SRTM (30m resolution)")
            logger.info("  - Land cover from ESA WorldCover (10m resolution)")
            spatial_features_list = self.gee.batch_get_spatial_features(coordinates)
        else:
            logger.info("\nUsing simulated spatial data (GEE not available)")
            spatial_features_list = []
            for lat, lon in coordinates:
                spatial_features_list.append(self.gee.get_spatial_features(lat, lon))
        
        # Predict at each grid point using REAL spatial features
        logger.info("\nPredicting communication metrics with ML models...")
        for i, point in enumerate(grid_points):
            point = self.predict_at_point(
                point, start_lat, start_lon, dest_lat, dest_lon,
                lora_params
            )
            point.distance_to_goal = self.calculate_distance(
                point.lat, point.lon, dest_lat, dest_lon
            )
        
        logger.info(f"Predictions completed for {len(grid_points)} points")
        
        # A* algorithm
        logger.info("\nRunning A* pathfinding algorithm...")
        start_node = grid_points[0]
        goal_node = grid_points[-1]
        
        open_set = [start_node]
        closed_set = set()
        came_from = {}
        g_score = {id(start_node): 0}
        f_score = {id(start_node): start_node.distance_to_goal}
        
        iterations = 0
        while open_set:
            iterations += 1
            current = min(open_set, key=lambda x: f_score.get(id(x), float('inf')))
            
            if current.grid_x == goal_node.grid_x and current.grid_y == goal_node.grid_y:
                # Reconstruct path
                path = [current]
                while id(current) in came_from:
                    current = came_from[id(current)]
                    path.insert(0, current)
                logger.info(f"Optimal path found in {iterations} iterations!")
                logger.info(f"Path has {len(path)} waypoints")
                
                # Exclude transmitter (first point) from path
                path = path[1:]  # Remove transmitter
                
                # Interpolate terrain between waypoints if enabled
                if config.interpolate_between_points and config.use_real_data:
                    path = self._refine_path_with_interpolation(
                        path, start_lat, start_lon, dest_lat, dest_lon, lora_params
                    )
                
                # Apply path smoothing if enabled
                if config.path_smoothing:
                    path = self.smooth_path(path)
                
                return path, grid_points
            
            open_set.remove(current)
            closed_set.add(id(current))
            
            # Get neighbors (8-connected grid)
            neighbors = [p for p in grid_points if
                        abs(p.grid_x - current.grid_x) <= 1 and
                        abs(p.grid_y - current.grid_y) <= 1 and
                        id(p) not in closed_set and
                        not (p.grid_x == current.grid_x and p.grid_y == current.grid_y)]
            
            for neighbor in neighbors:
                distance = self.calculate_distance(
                    current.lat, current.lon,
                    neighbor.lat, neighbor.lon
                )
                cost = self.calculate_lora_cost(neighbor, distance, lora_params)
                tentative_g_score = g_score[id(current)] + cost
                
                if id(neighbor) not in g_score or tentative_g_score < g_score[id(neighbor)]:
                    came_from[id(neighbor)] = current
                    g_score[id(neighbor)] = tentative_g_score
                    f_score[id(neighbor)] = tentative_g_score + neighbor.distance_to_goal / 10000
                    
                    if neighbor not in open_set:
                        open_set.append(neighbor)
        
        # If no path found, return direct path
        logger.warning("No optimal path found, returning direct path")
        return [start_node, goal_node], grid_points
    
    def _refine_path_with_interpolation(self, path: List[PathPoint], start_lat, start_lon, 
                                        dest_lat, dest_lon, lora_params: LoRaParameters):
        """Refine path by sampling terrain between waypoints"""
        logger.info("\nRefining path with micro-topography interpolation...")
        
        total_samples = 0
        refined_path = []
        
        for i in range(len(path)):
            current = path[i]
            refined_path.append(current)
            
            # Interpolate between current and next waypoint
            if i < len(path) - 1:
                next_point = path[i + 1]
                
                # Get interpolated features between these two points
                interp_features = self.interpolate_spatial_features(current, next_point, num_samples=5)
                
                total_samples += 5
                
                # Store interpolation info in current waypoint
                current.interp_to_next = {
                    'elevation_range': interp_features['elevation_range'],
                    'elevation_variance': interp_features['elevation_variance'],
                    'max_terrain_penalty': interp_features['max_terrain_penalty'],
                    'avg_elevation': interp_features['avg_elevation']
                }
                
                # If significant terrain variation detected, add warning
                if interp_features['elevation_range'] > 100:  # >100m elevation change
                    current.terrain_warning = 'High elevation variation detected'
                if interp_features['max_terrain_penalty'] > 0.5:
                    current.terrain_warning = current.get('terrain_warning', '') + ' High terrain penalty'
        
        logger.info(f"Sampled {total_samples} intermediate points between {len(path)} waypoints")
        logger.info(f"Total API calls: {len(path)} (grid) + {total_samples} (interpolation) = {len(path) + total_samples}")
        
        return refined_path


### 7. MultiHop Route Planning

In [ ]:
class MultiHopRouter:
    """Multi-hop routing for LoRa networks with real-time GEE data"""
    
    def __init__(self, model, scaler, feature_cols, gee_integration):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.gee = gee_integration
        self.optimizer = PathOptimizer(model, scaler, feature_cols, gee_integration)
        
        # Create output directory if it doesn't exist
        self.output_dir = Path("./output")
        self.output_dir.mkdir(exist_ok=True)
        
    def find_relay_nodes(self, start_lat, start_lon, dest_lat, dest_lon,
                          max_hop_distance_km: float, min_relay_distance_km: float,
                          lora_params: LoRaParameters,
                          config: OptimizationConfig):
        """Find optimal relay nodes for multi-hop routing"""
        total_distance = self.optimizer.calculate_distance(
            start_lat, start_lon, dest_lat, dest_lon
        )
        
        logger.info(f"Total distance: {total_distance/1000:.2f} km")
        logger.info(f"Max hop distance: {max_hop_distance_km} km")
        logger.info(f"Min relay distance: {min_relay_distance_km} km")
        
        # Calculate number of hops needed
        num_hops = int(np.ceil(total_distance / (max_hop_distance_km * 1000)))
        logger.info(f"Required hops: {num_hops}")
        
        if num_hops == 1:
            logger.info("Single hop sufficient, using standard path optimization...")
            return self.optimizer.find_optimal_path(
                start_lat, start_lon, dest_lat, dest_lon,
                lora_params, config
            )
        
        # Generate intermediate waypoints
        logger.info(f"\nGenerating {num_hops-1} intermediate relay points...")
        intermediate_points = []
        for i in range(1, num_hops):
            ratio = i / num_hops
            inter_lat = start_lat + (dest_lat - start_lat) * ratio
            inter_lon = start_lon + (dest_lon - start_lon) * ratio
            intermediate_points.append((inter_lat, inter_lon))
        
        # Optimize each hop
        all_hops = []
        current_start = (start_lat, start_lon)
        
        for i, next_point in enumerate(intermediate_points + [(dest_lat, dest_lon)]):
            logger.info(f"\nOptimizing hop {i+1}/{num_hops}...")
            logger.info(f"  From: ({current_start[0]:.6f}, {current_start[1]:.6f})")
            logger.info(f"  To: ({next_point[0]:.6f}, {next_point[1]:.6f})")
            
            hop_path, _ = self.optimizer.find_optimal_path(
                current_start[0], current_start[1],
                next_point[0], next_point[1],
                lora_params, config
            )
            
            # Mark relay nodes
            for point in hop_path:
                point.is_relay = True
                point.hop_number = i + 1
            
            all_hops.append({
                'hop_number': i+1,
                'path': hop_path,
                'start': current_start,
                'end': next_point
            })
            
            current_start = next_point
        
        logger.info(f"\nMulti-hop path completed with {num_hops} hops")
        return all_hops
    
    def visualize_multi_hop(self, hops, filename='multi_hop_routing.html'):
        """Visualize multi-hop routing"""
        logger.info(f"\nCreating multi-hop visualization: {filename}")
        
        # Calculate center point
        all_points = []
        for hop in hops:
            all_points.extend(hop['path'])
        
        all_lats = [p.lat for p in all_points]
        all_lons = [p.lon for p in all_points]
        center_lat = np.mean(all_lats)
        center_lon = np.mean(all_lons)
        
        # Create base map
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=12,
            tiles='OpenStreetMap'
        )
        
        # Color map for hops
        colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige']
        
        # Plot each hop
        for i, hop in enumerate(hops):
            path = hop['path']
            color = colors[i % len(colors)]
            
            # Create coordinates for the path
            coords = [[p.lat, p.lon] for p in path]
            
            # Add path
            folium.PolyLine(
                coords,
                color=color,
                weight=4,
                opacity=0.8,
                popup=f"Hop {hop['hop_number']}<br>PDR: {np.mean([p.pdr for p in path]):.3f}"
            ).add_to(m)
            
            # Add markers for relay nodes
            for j, point in enumerate(path):
                folium.Marker(
                    location=[point.lat, point.lon],
                    popup=f"Hop {hop['hop_number']} - Point {j+1}<br>"
                           f"PDR: {point.pdr:.3f}<br>"
                           f"RSSI: {point.rssi:.1f} dBm",
                    icon=folium.Icon(color=color, icon='info-sign')
                ).add_to(m)
        
        # Add start and end markers
        folium.Marker(
            location=[hops[0]['start'][0], hops[0]['start'][1]],
            popup="Start Point (Transmitter)",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)
        
        folium.Marker(
            location=[hops[-1]['end'][0], hops[-1]['end'][1]],
            popup="End Point (Receiver)",
            icon=folium.Icon(color='green', icon='stop')
        ).add_to(m)
        
        # Save map with error handling
        filepath = self.output_dir / filename
        try:
            m.save(str(filepath))
            logger.info(f"Multi-hop visualization saved to {filepath}")
        except Exception as e:
            logger.error(f"Error saving multi-hop visualization: {e}")
            # Try saving in the current directory as fallback
            try:
                m.save(filename)
                logger.info(f"Multi-hop visualization saved to current directory: {filename}")
            except Exception as e2:
                logger.error(f"Error saving multi-hop visualization to current directory: {e2}")


### 8. Coverage Map Generator

In [ ]:
class CoverageMapGenerator:
    """Generate coverage maps showing predicted signal quality in an area"""
    
    def __init__(self, model, scaler, feature_cols, gee_integration):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.gee = gee_integration
        
        # Create output directory if it doesn't exist
        self.output_dir = Path("./output")
        self.output_dir.mkdir(exist_ok=True)
        
    def generate_coverage_map(self, center_lat, center_lon, lora_params: LoRaParameters, 
                             radius_km=10, grid_resolution=50, filename='coverage_map.html'):
        """Generate a coverage map showing predicted signal quality"""
        logger.info("\n" + "="*60)
        logger.info("COVERAGE MAP GENERATION WITH REAL-TIME GEE DATA")
        logger.info("="*60)
        
        logger.info(f"\nGenerating {grid_resolution}x{grid_resolution} coverage map...")
        logger.info(f"Center: ({center_lat:.6f}, {center_lon:.6f})")
        logger.info(f"Radius: {radius_km} km")
        
        # Convert radius to degrees (approximate)
        lat_degree_km = 111.32
        lon_degree_km = 111.32 * np.cos(np.radians(center_lat))
        
        lat_range = radius_km / lat_degree_km
        lon_range = radius_km / lon_degree_km
        
        lats = np.linspace(center_lat - lat_range, center_lat + lat_range, grid_resolution)
        lons = np.linspace(center_lon - lon_range, center_lon + lon_range, grid_resolution)
        
        # Create meshgrid
        lat_grid, lon_grid = np.meshgrid(lats, lons)
        
        # Fetch spatial features
        coordinates = [(lat, lon) for lat, lon in zip(lat_grid.ravel(), lon_grid.ravel())]
        spatial_features_list = self.gee.batch_get_spatial_features(coordinates)
        
        # Prepare features and predict
        logger.info("\nPredicting coverage...")
        predictions_list = []
        
        for i, (lat, lon) in enumerate(coordinates):
            sf = spatial_features_list[i]
            
            # Distance from center (gateway location)
            distance = self._calculate_distance(center_lat, center_lon, lat, lon)
            
            features = np.array([[
                sf['elevation'],
                sf['land_cover'],
                sf['terrain_penalty'],
                0,  # distance_to_start
                distance,  # distance_to_destination
                lora_params.spreading_factor,
                lora_params.frequency,
                lora_params.tx_power,
                distance,  # distance_total
                sf['elevation'] / 1000  # elevation_normalized
            ]])
            
            features_scaled = self.scaler.transform(features)
            
            with torch.no_grad():
                X_tensor = torch.FloatTensor(features_scaled).to(device)
                pred = self.model(X_tensor).cpu().numpy()[0]
            
            predictions_list.append(pred)
        
        # Reshape predictions
        predictions_array = np.array(predictions_list)
        rssi_grid = predictions_array[:, 0].reshape(grid_resolution, grid_resolution)
        snr_grid = predictions_array[:, 1].reshape(grid_resolution, grid_resolution)
        pdr_grid = predictions_array[:, 2].reshape(grid_resolution, grid_resolution)
        
        elevation_grid = np.array([sf['elevation'] for sf in spatial_features_list]).reshape(grid_resolution, grid_resolution)
        
        # Create interactive map
        self._create_interactive_coverage_map(
            lat_grid, lon_grid, rssi_grid, snr_grid, pdr_grid, elevation_grid,
            center_lat, center_lon, filename
        )
        
        # Save coverage data
        coverage_data = pd.DataFrame({
            'latitude': lat_grid.ravel(),
            'longitude': lon_grid.ravel(),
            'elevation': elevation_grid.ravel(),
            'RSSI': rssi_grid.ravel(),
            'SNR': snr_grid.ravel(),
            'PDR': pdr_grid.ravel()
        })
        
        filepath = self.output_dir / 'coverage_map_data.csv'
        coverage_data.to_csv(filepath, index=False)
        logger.info(f"\nCoverage data saved to {filepath}")
        
        return coverage_data
    
    def _calculate_distance(self, lat1, lon1, lat2, lon2):
        """Calculate Haversine distance in meters"""
        R = 6371000
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        return R * c
    
    def _create_interactive_coverage_map(self, lat_grid, lon_grid, rssi_grid, snr_grid, pdr_grid, elevation_grid,
                                      center_lat, center_lon, filename):
        """Create interactive coverage map using Folium"""
        logger.info(f"\nCreating interactive coverage map: {filename}")
        
        # Create base map
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=11,
            tiles='OpenStreetMap'
        )
        
        # Create heatmap layers
        self._add_heatmap_layer(m, lat_grid, lon_grid, pdr_grid, 'PDR', ['green', 'yellow', 'red'], 0, 1)
        self._add_heatmap_layer(m, lat_grid, lon_grid, rssi_grid, 'RSSI', ['blue', 'white', 'red'], -150, -20)
        self._add_heatmap_layer(m, lat_grid, lon_grid, elevation_grid, 'Elevation', ['green', 'yellow', 'brown'], 0, 1000)
        
        # Add gateway marker
        folium.Marker(
            location=[center_lat, center_lon],
            popup="Gateway Location",
            icon=folium.Icon(color='red', icon='signal')
        ).add_to(m)
        
        # Add layer control
        folium.LayerControl().add_to(m)
        
        # Save map with error handling
        filepath = self.output_dir / filename
        try:
            m.save(str(filepath))
            logger.info(f"Coverage map saved to {filepath}")
        except Exception as e:
            logger.error(f"Error saving coverage map: {e}")
            # Try saving in the current directory as fallback
            try:
                m.save(filename)
                logger.info(f"Coverage map saved to current directory: {filename}")
            except Exception as e2:
                logger.error(f"Error saving coverage map to current directory: {e2}")
    
    def _add_heatmap_layer(self, map_obj, lat_grid, lon_grid, data_grid, name, colorscale, vmin, vmax):
        """Add a heatmap layer to the map"""
        # Flatten the grids
        lats = lat_grid.ravel()
        lons = lon_grid.ravel()
        data = data_grid.ravel()
        
        # Convert numpy types to Python native types for JSON serialization
        points = [[float(lat), float(lon), float(value)] for lat, lon, value in zip(lats, lons, data)]
        
        # Create heatmap
        from folium.plugins import HeatMap
        HeatMap(
            points,
            name=name,
            min_opacity=0.2,
            max_zoom=18,
            radius=25,
            blur=15,
            gradient={0.0: colorscale[0], 0.5: colorscale[1], 1.0: colorscale[2]}
        ).add_to(map_obj)


### 9. Visulization and Evaluation

### 10. Main Execution Pipeline